In [1]:
import os
import boto3
import getpass
from pathlib import Path
from typing import List

In [2]:
class InputConfig:
    """S3 설정"""
    REGION = "us-west-2"
    ENV = "dev"
    USER_ID = "sample" # <- 사용자 아이디(내가 데이터셋 저장한)
    PROJECT = "titanic-survival-prediction"
    EXPERIMENT = "baseline-awesome-sean-v1"
    VERSION = "v1.0"
    """S3 설정"""
    
    CONF_BUCKET = f"gs-retail-awesome-conf-{REGION}"
    DATA_BUCKET = f"gs-retail-awesome-data-{REGION}"
    
    @classmethod
    def get_conf_prefix(cls):
        # run_pm_2 구조: {env}/{user_id}/{project}/{experiment}
        return f"{cls.ENV}/{cls.USER_ID}/{cls.PROJECT}/{cls.EXPERIMENT}"
        
    @classmethod
    def get_data_prefix(cls):
        # run_pm_2 구조: {env}/{user_id}/{project}/{version}/data
        # data 파일들은 데이터 prefix 하위의 'data' 폴더에 저장됨
        return f"{cls.ENV}/{cls.USER_ID}/{cls.PROJECT}/{cls.VERSION}/data"

In [3]:
def download_from_s3(bucket: str, s3_prefix: str, local_dir: str) -> List[str]:
    """S3 버킷의 특정 Prefix 하위 모든 파일을 로컬로 다운로드"""
    s3_client = boto3.client('s3', region_name=InputConfig.REGION)
    downloaded = []
    
    # Prefix 경로 정규화
    full_prefix = s3_prefix.rstrip('/') + '/'
        
    try:
        paginator = s3_client.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=bucket, Prefix=full_prefix)
        
        for page in pages:
            for obj in page.get('Contents', []):
                s3_key = obj['Key']
                
                # 디렉토리 자체 객체는 무시
                if s3_key.endswith('/'):
                    continue
                    
                # 기준 s3_prefix로부터의 상대 경로 계산
                relative_path = s3_key[len(full_prefix):].lstrip('/')
                local_path = os.path.join(local_dir, relative_path)
                
                # 디렉토리 생성 및 다운로드
                os.makedirs(os.path.dirname(local_path), exist_ok=True)
                s3_client.download_file(bucket, s3_key, local_path)
                print(f"  ✓ Downloaded: s3://{bucket}/{s3_key} -> {local_path}")
                
                downloaded.append(local_path)
    except Exception as e:
        print(f"✗ Download error for s3://{bucket}/{full_prefix}: {e}")
            
    return downloaded

In [4]:
def download_inputs(local_root: str = "."):
    """
    modeling/titanic_modeling.ipynb에서 인풋(conf, data)으로 쓰이는 
    값들을 S3에서 각각 로컬 경로로 다운로드
    """
    local_base = Path(local_root)
    # titanic_modeling.ipynb의 인풋 인자로 쓰이는 디렉토리
    local_conf_dir = local_base / "conf"
    local_data_dir = local_base / "data"

    print("=" * 60)
    print("Step 0. local path:",local_base)
    print("=" * 60)
    
    conf_prefix = InputConfig.get_conf_prefix()
    data_prefix = InputConfig.get_data_prefix()
    
    print("=" * 60)
    print("Step 1. Conf 다운로드 (gs-retail-awesome-conf)")
    print("=" * 60)
    conf_downloaded = download_from_s3(
        bucket=InputConfig.CONF_BUCKET,
        s3_prefix=conf_prefix,
        local_dir=str(local_conf_dir)
    )
    
    print("\n" + "=" * 60)
    print("Step 2. Data 다운로드 (gs-retail-awesome-data)")
    print("=" * 60)
    data_downloaded = download_from_s3(
        bucket=InputConfig.DATA_BUCKET,
        s3_prefix=data_prefix,
        local_dir=str(local_data_dir)
    )
    
    print("\n" + "=" * 60)
    print(f"다운로드 완료!")
    print(f"- Conf 파일 {len(conf_downloaded)}개 -> {local_conf_dir}")
    print(f"- Data 파일 {len(data_downloaded)}개 -> {local_data_dir}")
    print("=" * 60)
    
    return {
        "conf_dir": local_conf_dir,
        "data_dir": local_data_dir,
        "conf_files": conf_downloaded,
        "data_files": data_downloaded
    }

In [5]:
download_inputs()

Step 0. local path: .
Step 1. Conf 다운로드 (gs-retail-awesome-conf)
  ✓ Downloaded: s3://gs-retail-awesome-conf-us-west-2/dev/sample/titanic-survival-prediction/baseline-awesome-sean-v1/env.yml -> conf/env.yml
  ✓ Downloaded: s3://gs-retail-awesome-conf-us-west-2/dev/sample/titanic-survival-prediction/baseline-awesome-sean-v1/meta.yml -> conf/meta.yml
  ✓ Downloaded: s3://gs-retail-awesome-conf-us-west-2/dev/sample/titanic-survival-prediction/baseline-awesome-sean-v1/model.yml -> conf/model.yml

Step 2. Data 다운로드 (gs-retail-awesome-data)
  ✓ Downloaded: s3://gs-retail-awesome-data-us-west-2/dev/sample/titanic-survival-prediction/v1.0/data/test.csv -> data/test.csv
  ✓ Downloaded: s3://gs-retail-awesome-data-us-west-2/dev/sample/titanic-survival-prediction/v1.0/data/train.csv -> data/train.csv
  ✓ Downloaded: s3://gs-retail-awesome-data-us-west-2/dev/sample/titanic-survival-prediction/v1.0/data/validation.csv -> data/validation.csv

다운로드 완료!
- Conf 파일 3개 -> conf
- Data 파일 3개 -> data


{'conf_dir': PosixPath('conf'),
 'data_dir': PosixPath('data'),
 'conf_files': ['conf/env.yml', 'conf/meta.yml', 'conf/model.yml'],
 'data_files': ['data/test.csv', 'data/train.csv', 'data/validation.csv']}